# 02. Tabular pipeline — preprocess → models → metrics

Тот же Titanic, но акцент на **единый пайплайн** и сравнение моделей.

💡 Так обычно выглядят аккуратные Kaggle baseline: один split, одинаковый preprocess, честное сравнение.

In [ ]:
# compare LogisticRegression vs RandomForest
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv(Path("../data/titanic.csv"))

# лёгкий feature engineering (типичный для Titanic)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

num_cols = ["Age", "Fare", "FamilySize", "IsAlone"]
cat_cols = ["Pclass", "Sex", "Embarked"]

X = df[num_cols + cat_cols]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
def make_pipe(model, scale_numeric=True):
    """Собрать preprocess + модель в один Pipeline."""
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])

candidates = {
    "logreg": make_pipe(LogisticRegression(max_iter=1000), scale_numeric=True),
    "rf": make_pipe(RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42), scale_numeric=False),
    "gb": make_pipe(GradientBoostingClassifier(random_state=42), scale_numeric=False),
}

rows = []
for name, pipe in candidates.items():
    # CV на train — оценка без подглядывания в test
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "cv_auc_mean": round(cv.mean(), 3),
        "cv_auc_std": round(cv.std(), 3),
        "test_auc": round(roc_auc_score(y_test, proba), 3),
    })

results = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
results

In [ ]:
results.set_index("model")[["cv_auc_mean", "test_auc"]].plot(kind="bar", title="AUC: CV vs holdout")
plt.ylabel("ROC-AUC")
plt.tight_layout()
plt.show()

print("best on test:", results.iloc[0].to_dict())
print("вывод: берём модель с лучшим test_auc при адекватном CV; не гонимся за 0.01 без смысла")